# Criação da Base Consolidada

Este notebook tem como objetivo construir a base inicial do projeto a partir dos arquivos brutos do SIH/SUS.

Os dados originais foram obtidos em formato `.dbc`, formato compactado utilizado pelo DATASUS. Para permitir a análise em Python, os arquivos são convertidos para `.dbf`, lidos como tabelas e posteriormente salvos em `.csv`.

Ao final do processo, todos os arquivos mensais são unidos em uma única base consolidada, que será utilizada nas próximas etapas do projeto, como análise exploratória, tratamento dos dados, engenharia de atributos e modelagem preditiva.

## Objetivo desta etapa

- Organizar os diretórios do projeto;
- Converter arquivos brutos `.dbc` para formatos manipuláveis em Python;
- Salvar os arquivos convertidos em `.csv`;
- Unificar os arquivos mensais em uma base única;
- Criar uma coluna de rastreabilidade indicando o arquivo de origem de cada registro.

## 1. Importação das bibliotecas

Nesta etapa são importadas as bibliotecas necessárias para manipulação de arquivos, leitura de dados e conversão dos arquivos originais do DATASUS.

A biblioteca `pathlib` é utilizada para trabalhar com caminhos de arquivos de forma mais organizada.  
O `pandas` será utilizado para carregar, transformar e consolidar os dados em DataFrames.  
A função `dbc2dbf` realiza a conversão dos arquivos `.dbc` para `.dbf`, enquanto `DBF` permite a leitura das tabelas convertidas.

In [1]:
# Manipulação de caminhos e arquivos
from pathlib import Path

# Manipulação e análise de dados
import pandas as pd

# Conversão de arquivos DATASUS
from dbfread import DBF
from pyreaddbc import dbc2dbf


## 2. Definição dos caminhos do projeto

Vou definir os caminhos das pastas utilizadas no processo de conversão.

A pasta `dados/raw` vai conter os arquivos brutos originais em formato DBC, enquanto a pasta `dados/interim` irá armazenar os arquivos intermediários gerados após a conversão.

Caso a pasta não exista, ela será criada.

In [2]:
# Pasta onde ficam os arquivos brutos baixados do SIH/SUS em formato DBC
pasta_raw = Path("dados/raw")

# Pasta onde serão armazenados os arquivos intermediários gerados no processo
pasta_interim = Path("dados/interim")

# Cria a pasta intermediária caso ela ainda não exista
pasta_interim.mkdir(parents=True, exist_ok=True)


## 3. Conversão dos arquivos DBC para CSV

Com isso feito, os arquivos DBC presentes na pasta `dados/raw` serão convertidos para o formato DBF, lidos como DataFrames e, então, cada arquivo será salvo individualmente em formato CSV na pasta `dados/interim`, facilitando a manipulação posterior, tendo em vista que o formato CSV é mais simples de carregar e analisar.

In [3]:
# Percorre todos os arquivos DBC disponíveis na pasta de dados brutos
for arquivo_dbc in pasta_raw.glob("*.dbc"):
    print(f"Convertendo {arquivo_dbc.name}...")

    # Define os caminhos de saída para o arquivo DBF temporário e para o CSV final
    arquivo_dbf = pasta_interim / f"{arquivo_dbc.stem}.dbf"
    arquivo_csv = pasta_interim / f"{arquivo_dbc.stem}.csv"

    # Converte o arquivo DBC original para DBF
    dbc2dbf(str(arquivo_dbc), str(arquivo_dbf))

    # Lê o arquivo DBF convertido, usando codificação compatível com os dados do DATASUS
    tabela = DBF(
        str(arquivo_dbf),
        encoding="latin1",
        char_decode_errors="ignore"
    )

    # Transforma a tabela DBF em um DataFrame pandas
    df_temp = pd.DataFrame(iter(tabela))

    # Salva o DataFrame em CSV para facilitar o carregamento nas próximas etapas
    df_temp.to_csv(
        arquivo_csv,
        index=False,
        sep=";",
        encoding="utf-8-sig"
    )

    print(f"Salvo: {arquivo_csv.name}")

# Lista todos os CSVs gerados a partir dos arquivos RD
arquivos_csv = list(Path("dados/interim").glob("RD*.csv"))


Convertendo RDMT2001.dbc...
Salvo: RDMT2001.csv
Convertendo RDMT2002.dbc...
Salvo: RDMT2002.csv
Convertendo RDMT2003.dbc...
Salvo: RDMT2003.csv
Convertendo RDMT2004.dbc...
Salvo: RDMT2004.csv
Convertendo RDMT2005.dbc...
Salvo: RDMT2005.csv
Convertendo RDMT2006.dbc...
Salvo: RDMT2006.csv
Convertendo RDMT2007.dbc...
Salvo: RDMT2007.csv
Convertendo RDMT2008.dbc...
Salvo: RDMT2008.csv
Convertendo RDMT2009.dbc...
Salvo: RDMT2009.csv
Convertendo RDMT2010.dbc...
Salvo: RDMT2010.csv
Convertendo RDMT2011.dbc...
Salvo: RDMT2011.csv
Convertendo RDMT2012.dbc...
Salvo: RDMT2012.csv
Convertendo RDMT2101.dbc...
Salvo: RDMT2101.csv
Convertendo RDMT2102.dbc...
Salvo: RDMT2102.csv
Convertendo RDMT2103.dbc...
Salvo: RDMT2103.csv
Convertendo RDMT2104.dbc...
Salvo: RDMT2104.csv
Convertendo RDMT2105.dbc...
Salvo: RDMT2105.csv
Convertendo RDMT2106.dbc...
Salvo: RDMT2106.csv
Convertendo RDMT2107.dbc...
Salvo: RDMT2107.csv
Convertendo RDMT2108.dbc...
Salvo: RDMT2108.csv
Convertendo RDMT2109.dbc...
Salvo: RDMT2

Após a conversão, os arquivos CSV serão carregados e unidos em um único DataFrame. Também será criada uma coluna indicando a origem do arquivo, permitindo-me identificar qual registro veio de qual arquivo, o que é útil para conferência, rastreabilidade e validação dos dados.

In [4]:
# Lista que irá armazenar temporariamente cada DataFrame mensal
dfs = []

# Percorre todos os arquivos CSV gerados na etapa anterior
for arquivo in arquivos_csv:
    # Lê cada CSV individualmente
    df_temp = pd.read_csv(arquivo, sep=";", encoding="utf-8-sig")

    # Registra o nome do arquivo de origem para manter rastreabilidade dos dados
    df_temp["arquivo_origem"] = arquivo.name

    # Adiciona o DataFrame mensal à lista
    dfs.append(df_temp)

# Consolida todos os arquivos mensais em uma única base
df = pd.concat(dfs, ignore_index=True)

# Verifica a dimensão da base consolidada
df.shape

# Define a pasta onde será armazenada a base final processada
pasta_processed = Path("dados/processed")
pasta_processed.mkdir(parents=True, exist_ok=True)


C:\Users\djmet\AppData\Local\Temp\ipykernel_28256\4039561158.py:7: DtypeWarning: Columns (0: DIAGSEC2, 1: DIAGSEC3, 2: DIAGSEC4, 3: DIAGSEC5) have mixed types. Specify dtype option on import or set low_memory=False.
  df_temp = pd.read_csv(arquivo, sep=";", encoding="utf-8-sig")
C:\Users\djmet\AppData\Local\Temp\ipykernel_28256\4039561158.py:10: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_temp["arquivo_origem"] = arquivo.name
C:\Users\djmet\AppData\Local\Temp\ipykernel_28256\4039561158.py:7: DtypeWarning: Columns (0: DIAGSEC2) have mixed types. Specify dtype option on import or set low_memory=False.
  df_temp = pd.read_csv(arquivo, sep=";", encoding="utf-8-sig")
C:\Users\djmet\AppData\Local\Temp\ipykernel_28256\4039561158.py:10: PerformanceWarning: Dat

Com isso feito, vou salvar a base consolidada. Essa será a base principal, utilizada nas próximas etapas do projeto.

In [5]:
# Caminho da base consolidada gerada na etapa anterior
caminho_base = Path("dados/processed/sih_sus_mt_internacoes_completo.csv")

# Carrega a base consolidada para conferência e uso nas próximas etapas
df = pd.read_csv(
    caminho_base,
    sep=";",
    encoding="utf-8-sig",
    low_memory=False
)

print(f"Arquivo carregado em: {caminho_base}")


Arquivo carregado em: dados\processed\sih_sus_mt_internacoes_completo.csv
